In [2]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import zipfile

import pandas as pd

# The notebook is located in the project's notebooks directory
PROJECT_ROOT = Path.cwd().parent

assert (PROJECT_ROOT / "PROJECT_PLAN.md").exists(), (
   "Make sure this notebook is saved in the project's notebooks directory"
)

ZIP_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "CCDB_Export_9_January_2025_through_February_2025.zip"
)

assert ZIP_PATH.exists(), "Downloaded file not found. Check the file path."

print("Project root:", PROJECT_ROOT)
print("ZIP size (bytes):", ZIP_PATH.stat().st_size)

Project root: /Users/celestine/Desktop/intern/Customer Churn Project/financial-complaint-intelligence
ZIP size (bytes): 75475199


In [3]:
sha256 = hashlib.sha256()

with ZIP_PATH.open("rb") as file:
    for chunk in iter(lambda: file.read(1024 * 1024), b""):
        sha256.update(chunk)

file_hash = sha256.hexdigest()

print("SHA-256:", file_hash)

SHA-256: 11c18bb0973eed8f5e4f66e2979346af0671a3aefcb5a6b9eec73009d53cff64


In [4]:
with zipfile.ZipFile(ZIP_PATH) as archive:
    members = [
        {
            "name": item.filename,
            "uncompressed_size_bytes": item.file_size,
        }
        for item in archive.infolist()
        if not item.is_dir()
    ]

manifest = {
    "dataset": "CFPB Consumer Complaint Narratives Archive",
    "source_page": (
        "https://www.consumerfinance.gov/foia-requests/"
        "foia-electronic-reading-room/"
        "cfpb-consumer-complaint-database-narratives-archive/"
    ),
    "download_url": (
        "https://files.consumerfinance.gov/f/documents/"
        "CCDB_Export_9_January_2025_through_February_2025.zip"
    ),
    "source_label": "January 2025 through February 2025",
    "local_file": ZIP_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "size_bytes": ZIP_PATH.stat().st_size,
    "sha256": file_hash,
    "archive_members": members,
    "manifest_created_at_utc": datetime.now(timezone.utc).isoformat(),
    "date_coverage_verified": False,
}

MANIFEST_PATH = (
    PROJECT_ROOT / "data" / "manifests" / "cfpb_2025_jan_feb.json"
)

MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print("Saved:", MANIFEST_PATH.relative_to(PROJECT_ROOT))

Saved: data/manifests/cfpb_2025_jan_feb.json


In [5]:
sample = pd.read_csv(
    ZIP_PATH,
    compression="zip",
    nrows=5,
    dtype="string",
)

print("Sample shape:", sample.shape)

print("\nColumns:")
for column in sample.columns:
    print("-", column)

Sample shape: (5, 16)

Columns:
- Date received
- Product
- Sub-product
- Issue
- Sub-issue
- Consumer complaint narrative
- Company public response
- Company
- State
- ZIP code
- Tags
- Submitted via
- Date sent to company
- Company response to consumer
- Timely response?
- Complaint ID


In [6]:
import shutil
import duckdb

CSV_NAME = "CCDB_Export_9_January_2025_through_February_2025.csv"
CSV_PATH = PROJECT_ROOT / "data" / "raw" / CSV_NAME
TEMP_PATH = CSV_PATH.with_suffix(".csv.part")

# Extract to a temporary file and rename only after extraction succeeds
with zipfile.ZipFile(ZIP_PATH) as archive:
    with archive.open(CSV_NAME) as source:
        with TEMP_PATH.open("wb") as destination:
            shutil.copyfileobj(source, destination, length=1024 * 1024)

TEMP_PATH.replace(CSV_PATH)

print("CSV file:", CSV_PATH.name)
print("CSV size (bytes):", CSV_PATH.stat().st_size)

CSV file: CCDB_Export_9_January_2025_through_February_2025.csv
CSV size (bytes): 491908736


In [7]:
con = duckdb.connect()

# Keep source fields as text during the initial audit
con.read_csv(
    str(CSV_PATH),
    header=True,
    all_varchar=True,
).create_view("complaints_raw", replace=True)

print("SQL view created: complaints_raw")

SQL view created: complaints_raw


In [8]:
audit_sql = """
WITH normalized AS (
    SELECT
        NULLIF(TRIM("Complaint ID"), '') AS complaint_id,
        NULLIF(TRIM("Date received"), '') AS received_raw,
        NULLIF(
            TRIM("Consumer complaint narrative"), ''
        ) AS narrative
    FROM complaints_raw
),
parsed AS (
    SELECT
        *,
        CAST(
            TRY_STRPTIME(
                received_raw,
                ['%Y-%m-%d', '%m/%d/%Y']
            ) AS DATE
        ) AS received_date
    FROM normalized
)
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT complaint_id) AS distinct_complaint_ids,
    COUNT(*) FILTER (
        WHERE complaint_id IS NULL
    ) AS missing_complaint_ids,
    MIN(received_date) AS earliest_received_date,
    MAX(received_date) AS latest_received_date,
    COUNT(*) FILTER (
        WHERE received_raw IS NULL
    ) AS missing_received_dates,
    COUNT(*) FILTER (
        WHERE received_raw IS NOT NULL
          AND received_date IS NULL
    ) AS unparsed_received_dates,
    COUNT(narrative) AS rows_with_narrative,
    ROUND(
        100.0 * COUNT(narrative) / NULLIF(COUNT(*), 0),
        2
    ) AS narrative_coverage_pct
FROM parsed
"""

audit_summary = con.execute(audit_sql).df()

display(audit_summary.T.rename(columns={0: "value"}))

,value
total_rows,811019
distinct_complaint_ids,811019
missing_complaint_ids,0
earliest_received_date,2025-01-01 00:00:00
latest_received_date,2025-02-28 00:00:00
missing_received_dates,0
unparsed_received_dates,0
rows_with_narrative,265405
narrative_coverage_pct,32.72


In [9]:
SUMMARY_PATH = PROJECT_ROOT / "reports" / "data_audit_summary.csv"
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)

audit_summary.to_csv(SUMMARY_PATH, index=False)

# Reload the saved manifest before updating it
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
row = audit_summary.iloc[0]

manifest["date_coverage_verified"] = True
manifest["observed_date_coverage"] = {
    "earliest_received_date": row["earliest_received_date"].strftime("%Y-%m-%d"),
    "latest_received_date": row["latest_received_date"].strftime("%Y-%m-%d"),
    "missing_received_dates": int(row["missing_received_dates"]),
    "unparsed_received_dates": int(row["unparsed_received_dates"]),
}
manifest["observed_counts"] = {
    "total_rows": int(row["total_rows"]),
    "distinct_complaint_ids": int(row["distinct_complaint_ids"]),
    "rows_with_narrative": int(row["rows_with_narrative"]),
}

MANIFEST_PATH.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print("Saved:", SUMMARY_PATH.relative_to(PROJECT_ROOT))
print("Updated:", MANIFEST_PATH.relative_to(PROJECT_ROOT))

Saved: reports/data_audit_summary.csv
Updated: data/manifests/cfpb_2025_jan_feb.json


In [10]:
product_sql = """
WITH normalized AS (
    SELECT
        NULLIF(TRIM("Product"), '') AS product,
        NULLIF(
            TRIM("Consumer complaint narrative"), ''
        ) AS narrative
    FROM complaints_raw
)
SELECT
    product,
    COUNT(*) AS total_rows,
    COUNT(narrative) AS rows_with_narrative,
    ROUND(
        100.0 * COUNT(narrative) / COUNT(*),
        2
    ) AS narrative_coverage_pct
FROM normalized
GROUP BY product
ORDER BY total_rows DESC
"""

product_summary = con.execute(product_sql).df()

display(product_summary)

product_summary.to_csv(
    PROJECT_ROOT / "reports" / "product_summary.csv",
    index=False,
)

,product,total_rows,rows_with_narrative,narrative_coverage_pct
0,Credit reporting or other personal consumer re...,659700,170863,25.90
1,"Money transfer, virtual currency, or money ser...",55871,46258,82.79
2,Debt collection,40862,15288,37.41
3,Checking or savings account,23819,17070,71.67
4,Credit card,15286,6958,45.52
5,Student loan,4100,2201,53.68
6,Mortgage,3958,2330,58.87
7,Vehicle loan or lease,3278,1910,58.27
8,"Payday loan, title loan, personal loan, or adv...",1848,1121,60.66
9,Prepaid card,1251,747,59.71


In [11]:
credit_card_view_sql = """
CREATE OR REPLACE VIEW credit_card_raw AS
SELECT *
FROM complaints_raw
WHERE TRIM("Product") = 'Credit card'
"""

con.execute(credit_card_view_sql)

print("SQL view created: credit_card_raw")

SQL view created: credit_card_raw


In [12]:
issue_sql = """
WITH normalized AS (
    SELECT
        NULLIF(TRIM("Issue"), '') AS issue,
        NULLIF(
            TRIM("Consumer complaint narrative"), ''
        ) AS narrative
    FROM credit_card_raw
)
SELECT
    issue,
    COUNT(*) AS total_rows,
    COUNT(narrative) AS rows_with_narrative,
    ROUND(
        100.0 * COUNT(narrative) / COUNT(*),
        2
    ) AS narrative_coverage_pct,
    ROUND(
        100.0 * COUNT(narrative)
        / NULLIF(SUM(COUNT(narrative)) OVER (), 0),
        2
    ) AS share_of_available_text_pct
FROM normalized
GROUP BY issue
ORDER BY rows_with_narrative DESC
"""

issue_summary = con.execute(issue_sql).df()

# Print full category names without display truncation
print(issue_summary.to_string(index=False))

print("\nTotal credit card complaints:",
      int(issue_summary["total_rows"].sum()))

print("Total credit card complaints with text:",
      int(issue_summary["rows_with_narrative"].sum()))

issue_summary.to_csv(
    PROJECT_ROOT / "reports" / "credit_card_issue_summary.csv",
    index=False,
)

                                                          issue  total_rows  rows_with_narrative  narrative_coverage_pct  share_of_available_text_pct
                Problem with a purchase shown on your statement        2775                 1628                   58.67                        23.40
                                               Fees or interest        1571                  984                   62.64                        14.14
                             Other features, terms, or problems        1433                  855                   59.67                        12.29
                                          Getting a credit card        1896                  820                   43.25                        11.78
                                           Closing your account         962                  576                   59.88                         8.28
Problem with a company's investigation into an existing problem        2724                  504    

In [13]:
text_view_sql = r"""
CREATE OR REPLACE VIEW credit_card_text AS
WITH normalized AS (
    SELECT
        NULLIF(TRIM("Complaint ID"), '') AS complaint_id,
        NULLIF(TRIM("Issue"), '') AS issue,
        "Date received" AS received_raw,
        NULLIF(
            TRIM("Consumer complaint narrative"), ''
        ) AS narrative
    FROM credit_card_raw
)
SELECT
    *,
    LOWER(
        REGEXP_REPLACE(narrative, '\s+', ' ', 'g')
    ) AS text_key
FROM normalized
WHERE narrative IS NOT NULL
"""

con.execute(text_view_sql)

print("SQL view created: credit_card_text")

SQL view created: credit_card_text


In [14]:
length_sql = """
SELECT
    COUNT(*) AS text_rows,
    COUNT(*) FILTER (
        WHERE issue IS NULL
    ) AS missing_issue_rows,
    MIN(LENGTH(narrative)) AS min_chars,
    ROUND(AVG(LENGTH(narrative)), 1) AS mean_chars,
    QUANTILE_CONT(LENGTH(narrative), 0.50) AS median_chars,
    QUANTILE_CONT(LENGTH(narrative), 0.90) AS p90_chars,
    QUANTILE_CONT(LENGTH(narrative), 0.99) AS p99_chars,
    MAX(LENGTH(narrative)) AS max_chars,
    COUNT(*) FILTER (
        WHERE LENGTH(narrative) <= 50
    ) AS rows_at_most_50_chars
FROM credit_card_text
"""

length_summary = con.execute(length_sql).df()

display(length_summary.T.rename(columns={0: "value"}))

length_summary.to_csv(
    PROJECT_ROOT / "reports" / "credit_card_text_lengths.csv",
    index=False,
)

,value
text_rows,6958.00
missing_issue_rows,0.00
min_chars,49.00
mean_chars,1139.10
median_chars,777.50
p90_chars,2405.00
p99_chars,5672.51
max_chars,31657.00
rows_at_most_50_chars,4.00


In [15]:
duplicate_sql = """
WITH text_groups AS (
    SELECT
        text_key,
        COUNT(*) AS group_size,
        COUNT(DISTINCT issue) AS distinct_issue_labels
    FROM credit_card_text
    GROUP BY text_key
)
SELECT
    COUNT(*) AS distinct_text_keys,
    COUNT(*) FILTER (
        WHERE group_size > 1
    ) AS repeated_text_groups,
    COALESCE(
        SUM(group_size) FILTER (WHERE group_size > 1),
        0
    ) AS rows_in_repeated_groups,
    COALESCE(
        SUM(group_size - 1),
        0
    ) AS excess_repeated_rows,
    COUNT(*) FILTER (
        WHERE group_size > 1
          AND distinct_issue_labels > 1
    ) AS repeated_groups_with_label_conflicts
FROM text_groups
"""

duplicate_summary = con.execute(duplicate_sql).df()

display(duplicate_summary.T.rename(columns={0: "value"}))

duplicate_summary.to_csv(
    PROJECT_ROOT / "reports" / "credit_card_text_duplicates.csv",
    index=False,
)

,value
distinct_text_keys,6480.0
repeated_text_groups,74.0
rows_in_repeated_groups,552.0
excess_repeated_rows,478.0
repeated_groups_with_label_conflicts,6.0


In [16]:
duplicate_groups_sql = """
SELECT
    MD5(text_key) AS text_group_id,
    COUNT(*) AS group_size,
    COUNT(DISTINCT issue) AS distinct_issue_labels,
    STRING_AGG(
        DISTINCT issue,
        ' | ' ORDER BY issue
    ) AS issue_labels,
    MIN(LENGTH(narrative)) AS min_chars,
    MAX(LENGTH(narrative)) AS max_chars
FROM credit_card_text
GROUP BY text_key
HAVING COUNT(*) > 1
ORDER BY group_size DESC, text_group_id
"""

duplicate_groups = con.execute(duplicate_groups_sql).df()

print("Largest repeated groups:")
print(duplicate_groups.head(10).to_string(index=False))

print("\nRepeated groups with label conflicts:")
print(
    duplicate_groups.loc[
        duplicate_groups["distinct_issue_labels"] > 1
    ].to_string(index=False)
)

duplicate_groups.to_csv(
    PROJECT_ROOT / "reports" / "credit_card_duplicate_groups.csv",
    index=False,
)

Largest repeated groups:
                   text_group_id  group_size  distinct_issue_labels                                                                                           issue_labels  min_chars  max_chars
6fa05b0b11c1b3f6163c005ee8cfcd2d          74                      1                                                                                  Getting a credit card        182        182
b6448b4a3e018926dcec2907ce30db90          71                      2 Incorrect information on your report | Problem with a company's investigation into an existing problem        275        275
0683cf6afaca25e2762510f7044fd839          51                      1                                        Problem with a company's investigation into an existing problem        276        276
f70c453814509a016e066203ca4cb0a5          29                      1                                        Problem with a company's investigation into an existing problem        261        261
35a37dbeff

In [17]:
short_cases = con.execute("""
    SELECT
        complaint_id,
        issue,
        LENGTH(narrative) AS chars,
        narrative
    FROM credit_card_text
    WHERE LENGTH(narrative) <= 50
    ORDER BY chars, complaint_id
""").df()

for case in short_cases.itertuples(index=False):
    print("Complaint ID:", case.complaint_id)
    print("Issue:", case.issue)
    print("Characters:", case.chars)
    print("Text:", case.narrative)
    print("-" * 60)

Complaint ID: 11843016
Issue: Fees or interest
Characters: 49
Text: XXXX and XXXX keeps taking out money from my card
------------------------------------------------------------
Complaint ID: 11585355
Issue: Fees or interest
Characters: 50
Text: NFCU charged too many fees and interest from XXXX.
------------------------------------------------------------
Complaint ID: 11662252
Issue: Fees or interest
Characters: 50
Text: Not accepting payments and overcharge on interest.
------------------------------------------------------------
Complaint ID: 12037293
Issue: Getting a credit card
Characters: 50
Text: I got a capital one platinum card I isnt apply for
------------------------------------------------------------


In [18]:
conflict_cases = con.execute("""
    SELECT
        MD5(text_key) AS text_group_id,
        COUNT(*) AS group_size,
        STRING_AGG(
            DISTINCT issue,
            ' | ' ORDER BY issue
        ) AS issue_labels,
        MIN(narrative) AS representative_text
    FROM credit_card_text
    GROUP BY text_key
    HAVING COUNT(DISTINCT issue) > 1
    ORDER BY group_size DESC, text_group_id
""").df()

for case in conflict_cases.itertuples(index=False):
    print("Text group:", case.text_group_id)
    print("Group size:", case.group_size)
    print("Issue labels:", case.issue_labels)
    print("Text preview:", case.representative_text[:800])
    print("-" * 60)

Text group: b6448b4a3e018926dcec2907ce30db90
Group size: 71
Issue labels: Incorrect information on your report | Problem with a company's investigation into an existing problem
Text preview: I need your assistance in reviewing accounts from companies that are incorrectly reporting late payments. After checking my credit report, I found that certain accounts have late payment marks that should not be there. These accounts should not show any late payment remarks.
------------------------------------------------------------
Text group: 35a37dbeffaf64b1ec28389de8556301
Group size: 26
Issue labels: Incorrect information on your report | Problem with a company's investigation into an existing problem
Text preview: I need your assistance in reviewing accounts from companies that have incorrectly reported late payments. After checking my credit report, I found that some accounts were marked late in error. There should be no late payment remarks on these accounts.
-----------------------------

In [19]:
quality_view_sql = """
CREATE OR REPLACE VIEW credit_card_quality AS
WITH group_stats AS (
    SELECT
        text_key,
        COUNT(*) AS text_group_size,
        COUNT(DISTINCT issue) AS distinct_issue_labels
    FROM credit_card_text
    GROUP BY text_key
)
SELECT
    t.complaint_id,
    t.issue,
    t.received_raw,
    t.narrative,
    MD5(t.text_key) AS text_group_id,
    LENGTH(t.narrative) AS narrative_chars,
    g.text_group_size,
    g.distinct_issue_labels,
    LENGTH(t.narrative) <= 50 AS is_short_text,
    g.text_group_size > 1 AS is_repeated_text,
    g.distinct_issue_labels > 1 AS has_label_conflict
FROM credit_card_text AS t
JOIN group_stats AS g
    ON t.text_key = g.text_key
"""

con.execute(quality_view_sql)

print("SQL view created: credit_card_quality")

SQL view created: credit_card_quality


In [20]:
quality_summary = con.execute("""
    SELECT
        COUNT(*) AS total_text_rows,
        COUNT(*) FILTER (
            WHERE is_short_text
        ) AS short_text_rows,
        COUNT(*) FILTER (
            WHERE is_repeated_text
        ) AS repeated_text_rows,
        COUNT(*) FILTER (
            WHERE has_label_conflict
        ) AS label_conflict_rows,
        COUNT(DISTINCT text_group_id) FILTER (
            WHERE has_label_conflict
        ) AS label_conflict_groups
    FROM credit_card_quality
""").df()

display(quality_summary.T.rename(columns={0: "value"}))

quality_summary.to_csv(
    PROJECT_ROOT / "reports" / "credit_card_quality_summary.csv",
    index=False,
)

,value
total_text_rows,6958
short_text_rows,4
repeated_text_rows,552
label_conflict_rows,112
label_conflict_groups,6


In [21]:
quality_flags = con.execute("""
    SELECT
        complaint_id,
        issue,
        received_raw,
        text_group_id,
        narrative_chars,
        text_group_size,
        distinct_issue_labels,
        is_short_text,
        is_repeated_text,
        has_label_conflict
    FROM credit_card_quality
    ORDER BY complaint_id
""").df()

FLAGS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "credit_card_quality_flags.parquet"
)

FLAGS_PATH.parent.mkdir(parents=True, exist_ok=True)
quality_flags.to_parquet(FLAGS_PATH, index=False)

print("Saved:", FLAGS_PATH.relative_to(PROJECT_ROOT))
print("Rows:", len(quality_flags))

Saved: data/processed/credit_card_quality_flags.parquet
Rows: 6958


In [22]:
SQL_DIR = PROJECT_ROOT / "sql"
SQL_DIR.mkdir(parents=True, exist_ok=True)

sql_queries = {
    "01_initial_audit.sql": audit_sql,
    "02_product_summary.sql": product_sql,
    "03_credit_card_view.sql": credit_card_view_sql,
    "04_credit_card_issue_summary.sql": issue_sql,
    "05_credit_card_text_view.sql": text_view_sql,
    "06_credit_card_text_lengths.sql": length_sql,
    "07_credit_card_text_duplicates.sql": duplicate_sql,
    "08_credit_card_duplicate_groups.sql": duplicate_groups_sql,
    "09_credit_card_quality_view.sql": quality_view_sql,
}

for filename, query in sql_queries.items():
    path = SQL_DIR / filename
    path.write_text(query.strip() + "\n", encoding="utf-8")

print("SQL files saved:", len(sql_queries))

SQL files saved: 9


In [23]:
audit_report = """# Initial CFPB Data Audit

## Scope

This pilot audit uses the official CFPB archive labeled
January 2025 through February 2025.

The observed complaint receipt dates range from 2025-01-01
through 2025-02-28. This verifies receipt-date coverage,
not historical public availability.

## Dataset Overview

- Total records: 811,019
- Unique complaint IDs: 811,019
- Missing complaint IDs: 0
- Missing or unparsed receipt dates: 0
- Records with non-empty narratives: 265,405 (32.72%)

## Credit Card Subset

The exact product value used for filtering is `Credit card`.

- Total credit card complaints: 15,286
- Complaints with non-empty narratives: 6,958 (45.52%)
- Observed issue categories: 15
- Missing issue labels among text records: 0

Category counts are imbalanced. Narrative availability also
varies across categories. The text subset should not be
treated as representative of all credit card complaints.

## Text Quality

- Median narrative length: 777.5 characters
- Maximum narrative length: 31,657 characters
- Narratives with at most 50 characters: 4
- Distinct normalized text keys: 6,480
- Repeated text groups: 74
- Records in repeated groups: 552
- Excess repeated records: 478
- Groups with conflicting issue labels: 6
- Records in conflicting-label groups: 112

Text matching lowercases narratives and collapses whitespace.
It retains numbers, punctuation, and redaction placeholders.

## Manual Review and Current Decisions

The four short narratives contain substantive complaint information.
They are retained.

Some conflicting-label groups use generic wording or describe
overlapping issues. Original labels are retained and conflicts
are flagged rather than automatically corrected.

Two reviewed groups have similar wording but different text keys.
Exact normalized matching therefore does not capture all near-duplicates.

Repeated complaint text does not establish that the underlying
complaint events are duplicates. No records have been deleted.

Consumer allegations are not treated as independently verified facts.

## Next Steps

- Expand the historical date range.
- Repeat integrity and label checks across all source files.
- Investigate cross-file exact and near-duplicate text.
- Define chronological splits and duplicate-group handling before modeling.
- Reassess rare categories using the expanded data.
- Evaluate model-token lengths before choosing truncation settings.

## Reproduction

Run `notebooks/01_data_audit.ipynb` from top to bottom.
The source archive metadata is recorded in
`data/manifests/cfpb_2025_jan_feb.json`.

Raw data and row-level derived files remain local.
Aggregate results are saved in `reports/`, and SQL queries in `sql/`.

No classifier has been trained and no business impact has been measured.
"""

REPORT_PATH = PROJECT_ROOT / "reports" / "data_audit.md"
REPORT_PATH.write_text(audit_report, encoding="utf-8")

print("Saved:", REPORT_PATH.relative_to(PROJECT_ROOT))

Saved: reports/data_audit.md
